In [1]:
# ============================================
# CELDA 1 — CONFIGURACIÓN GEOGRÁFICA DE LA ESTACIÓN IDEAM
# ============================================

ESTACION = "Santa Marta"

LATITUD_ESTACION = 11.114772
LONGITUD_ESTACION = -74.231028

In [2]:
### Celda 2 — EXTRACCIÓN DE LA SERIE TEMPORAL DEL PÍXEL ERA5-LAND MÁS CERCANO
import os
import numpy as np
import pandas as pd
import xarray as xr

# ==============================================================================
# PARÁMETROS DE ENTRADA
# ==============================================================================

# Carpeta donde se encuentran almacenados los archivos NetCDF de ERA5-Land
ERA5_FOLDER = "ERA5_NC"

# ==============================================================================
# IDENTIFICACIÓN DE LOS ARCHIVOS NETCDF DISPONIBLES
# ==============================================================================

archivos_nc = sorted(
    [
        archivo
        for archivo in os.listdir(ERA5_FOLDER)
        if archivo.endswith(".nc")
    ]
)

if len(archivos_nc) == 0:
    raise FileNotFoundError(
        f"No se encontraron archivos NetCDF en:\n{ERA5_FOLDER}"
    )

print("=" * 80)
print("EXTRACCIÓN DE LA SERIE TEMPORAL ERA5-LAND")
print("=" * 80)
print(f"Estación analizada : {ESTACION}")
print(f"Archivos procesados: {len(archivos_nc)}")

# ==============================================================================
# EXTRACCIÓN DEL PÍXEL MÁS CERCANO PARA CADA ARCHIVO
# ==============================================================================

# Lista donde se almacenará la serie temporal extraída de cada archivo
era5_frames = []

for archivo in archivos_nc:

    print(f"Procesando: {archivo}")

    ruta_nc = os.path.join(ERA5_FOLDER, archivo)

    # Apertura del archivo mediante lazy loading para optimizar memoria
    ds = xr.open_dataset(ruta_nc)

    # Identificación automática del nombre de las coordenadas
    if "latitude" in ds.coords:
        lat_name = "latitude"
        lon_name = "longitude"
    else:
        lat_name = "lat"
        lon_name = "lon"

    # Selección del píxel más cercano a la estación mediante vecino más próximo
    ds_pixel = ds.sel(
        {
            lat_name: LATITUD_ESTACION,
            lon_name: LONGITUD_ESTACION
        },
        method="nearest"
    )

    # Conversión únicamente del píxel seleccionado a DataFrame
    era5_frames.append(
        ds_pixel.to_dataframe().reset_index()
    )

    # Liberación del archivo de memoria
    ds.close()

# ==============================================================================
# CONSOLIDACIÓN DE LA SERIE TEMPORAL
# ==============================================================================

# Unión de la información proveniente de todos los archivos NetCDF
df_era5_estacion = pd.concat(
    era5_frames,
    ignore_index=True
)

# ==============================================================================
# NORMALIZACIÓN DE LA VARIABLE TEMPORAL
# ==============================================================================

# Se crea una única columna temporal denominada "valid_time", independientemente
# del nombre utilizado en el archivo NetCDF.
for columna in ["valid_time", "time", "date"]:

    if columna in df_era5_estacion.columns:

        df_era5_estacion["valid_time"] = pd.to_datetime(
            df_era5_estacion[columna]
        )

        break

else:

    raise ValueError(
        "No se encontró una columna temporal válida."
    )

# ==============================================================================
# VALIDACIÓN DEL PÍXEL SELECCIONADO
# ==============================================================================

pixel_lat = float(df_era5_estacion[lat_name].iloc[0])
pixel_lon = float(df_era5_estacion[lon_name].iloc[0])

# Distancia aproximada entre la estación y el píxel seleccionado
distancia_km = np.sqrt(
    ((LATITUD_ESTACION - pixel_lat) * 111) ** 2 +
    (
        (LONGITUD_ESTACION - pixel_lon)
        * 111
        * np.cos(np.radians(LATITUD_ESTACION))
    ) ** 2
)

# ==============================================================================
# RESUMEN DEL PROCESAMIENTO
# ==============================================================================

print("\n" + "=" * 80)
print("RESUMEN DE LA EXTRACCIÓN")
print("=" * 80)

print(f"Período temporal      : {df_era5_estacion['valid_time'].min()}  →  {df_era5_estacion['valid_time'].max()}")
print(f"Registros horarios    : {len(df_era5_estacion):,}")
print(f"Variables disponibles : {df_era5_estacion.shape[1]}")

print("\nCoordenadas de la estación IDEAM")
print(f"Latitud              : {LATITUD_ESTACION:.6f}")
print(f"Longitud             : {LONGITUD_ESTACION:.6f}")

print("\nCoordenadas del píxel ERA5-Land seleccionado")
print(f"Latitud              : {pixel_lat:.6f}")
print(f"Longitud             : {pixel_lon:.6f}")

print(f"\nDistancia aproximada estación-píxel: {distancia_km:.2f} km")

print("=" * 80)

display(df_era5_estacion.head())

EXTRACCIÓN DE LA SERIE TEMPORAL ERA5-LAND
Estación analizada : Santa Marta
Archivos procesados: 892
Procesando: 0001.nc
Procesando: 0002.nc
Procesando: 0003.nc
Procesando: 0004.nc
Procesando: 0005.nc
Procesando: 0006.nc
Procesando: 0007.nc
Procesando: 0008.nc
Procesando: 0009.nc
Procesando: 0010.nc
Procesando: 0011.nc
Procesando: 0012.nc
Procesando: 0101.nc
Procesando: 0102.nc
Procesando: 0103.nc
Procesando: 0104.nc
Procesando: 0105.nc
Procesando: 0106.nc
Procesando: 0107.nc
Procesando: 0108.nc
Procesando: 0109.nc
Procesando: 0110.nc
Procesando: 0111.nc
Procesando: 0112.nc
Procesando: 0201.nc
Procesando: 0202.nc
Procesando: 0203.nc
Procesando: 0204.nc
Procesando: 0205.nc
Procesando: 0206.nc
Procesando: 0207.nc
Procesando: 0208.nc
Procesando: 0209.nc
Procesando: 0210.nc
Procesando: 0211.nc
Procesando: 0212.nc
Procesando: 0301.nc
Procesando: 0302.nc
Procesando: 0303.nc
Procesando: 0304.nc
Procesando: 0305.nc
Procesando: 0306.nc
Procesando: 0307.nc
Procesando: 0308.nc
Procesando: 0309.nc


,valid_time,d2m,t2m,ssrd,e,u10,v10,sp,tp,number,latitude,longitude,expver
0,2000-01-01 00:00:00,294.532990,297.480682,13998488.0,-0.003094,-5.205154,-4.128635,97178.8750,0.001137,0,11.1,-74.2,0001
1,2000-01-01 01:00:00,294.427185,297.559692,0.0,-0.000059,-5.404922,-3.954248,97229.3125,0.000010,0,11.1,-74.2,0001
2,2000-01-01 02:00:00,294.188232,297.689301,0.0,-0.000124,-5.255600,-3.664094,97278.3750,0.000026,0,11.1,-74.2,0001
3,2000-01-01 03:00:00,293.882233,297.770569,0.0,-0.000192,-5.089340,-3.449065,97326.0000,0.000051,0,11.1,-74.2,0001
4,2000-01-01 04:00:00,293.576416,297.786377,0.0,-0.000265,-4.881149,-3.127121,97318.3125,0.000081,0,11.1,-74.2,0001


In [3]:
### Celda 3 — ERA5-Land horario → diario (día local Santa Marta, UTC-5)

import pandas as pd

df_era5_diario = df_era5_estacion[["valid_time", "tp"]].copy()
df_era5_diario["valid_time"] = pd.to_datetime(df_era5_diario["valid_time"])
df_era5_diario = df_era5_diario.sort_values("valid_time").reset_index(drop=True)

# 1. Duplicados horarios: se detienen para revisión, no se descartan
duplicados = df_era5_diario.duplicated(subset="valid_time").sum()
if duplicados > 0:
    raise ValueError(f"Se detectaron {duplicados} registros horarios duplicados.")

# 2. Verificación del reinicio de ciclo de acumulación (documentado por ECMWF):
#    el decrecimiento debe ocurrir únicamente en el primer paso tras el cierre (hora 01)
serie_tp = df_era5_diario.set_index("valid_time")["tp"]
diffs = serie_tp.diff()
decrecimientos_inesperados = ((diffs < -1e-6) & (serie_tp.index.hour != 1)).sum()
print(f"Decrecimientos fuera del reinicio esperado (hora≠01): {decrecimientos_inesperados}")

# 3. Reconstrucción del día local: 00:00 local(D) → 00:00 local(D+1) = 05:00 UTC(D) → 05:00 UTC(D+1)
#    tp_diaria = TP_00(D+1) - TP_05(D) + TP_05(D+1)
df_era5_diario["Fecha"] = df_era5_diario["valid_time"].dt.normalize()
df_era5_diario["Hora"] = df_era5_diario["valid_time"].dt.hour

df_pivot = df_era5_diario[df_era5_diario["Hora"].isin([0, 5])].pivot(
    index="Fecha", columns="Hora", values="tp"
).rename(columns={0: "TP_00", 5: "TP_05"})

df_pivot["TP_00_siguiente"] = df_pivot["TP_00"].shift(-1)
df_pivot["TP_05_siguiente"] = df_pivot["TP_05"].shift(-1)
df_pivot["tp_diaria_m"] = (
    df_pivot["TP_00_siguiente"] - df_pivot["TP_05"] + df_pivot["TP_05_siguiente"]
)

filas_totales = len(df_pivot)
df_pivot = df_pivot.dropna(subset=["tp_diaria_m"])
print(f"Días reconstruidos: {len(df_pivot)} de {filas_totales} "
      f"({filas_totales - len(df_pivot)} excluidos por discontinuidades horarias)")

# 4. Diagnóstico de negativos antes de recortar (no se ocultan sin reportar)
negativos = df_pivot.loc[df_pivot["tp_diaria_m"] < 0, "tp_diaria_m"]
if len(negativos) > 0:
    print(f"Valores negativos antes de recorte: {len(negativos)} "
          f"(min={negativos.min():.6f}, max={negativos.max():.6f}) [unidades: m]")

df_era5_diario = (
    df_pivot.assign(tp=lambda x: x["tp_diaria_m"].clip(lower=0) * 1000)
    .reset_index()[["Fecha", "tp"]]
    .rename(columns={"Fecha": "Fecha_Diaria"})
    .reset_index(drop=True)
)

display(df_era5_diario.head())
display(df_era5_diario.tail())

Decrecimientos fuera del reinicio esperado (hora≠01): 0
Días reconstruidos: 27120 de 27121 (1 excluidos por discontinuidades horarias)
Valores negativos antes de recorte: 1 (min=-0.000000, max=-0.000000) [unidades: m]


Hora,Fecha_Diaria,tp
0,1952-06-01,46.130993
1,1952-06-02,3.012763
2,1952-06-03,5.734891
3,1952-06-04,18.395502
4,1952-06-05,24.197504


Hora,Fecha_Diaria,tp
27115,2026-08-27,1.041339
27116,2026-08-28,1.216486
27117,2026-08-29,3.136174
27118,2026-08-30,0.652130
27119,2026-08-31,18.834227


In [4]:
# ### Celda 4 — CARGA DE DATOS IDEAM

import pandas as pd

# Cargar archivo
df_ideam = pd.read_csv("IDEAM/prec_estacion.csv")

# Conservar únicamente las variables de interés
df_ideam = df_ideam[["Fecha", "Valor"]]

# Convertir Fecha a formato fecha
df_ideam["Fecha"] = pd.to_datetime(df_ideam["Fecha"])

display(df_ideam.head())
display(df_ideam.tail())

,Fecha,Valor
0,1952-06-01,0.0
1,1952-06-02,0.0
2,1952-06-03,0.0
3,1952-06-04,0.0
4,1952-06-05,0.0


,Fecha,Valor
23260,2026-08-22,0.3
23261,2026-08-23,34.4
23262,2026-08-25,2.0
23263,2026-08-30,1.8
23264,2026-08-31,0.5


In [5]:
### Celda 5 — INTEGRACIÓN ERA5-LAND + IDEAM

df_consolidado = pd.merge(
    df_era5_diario,
    df_ideam,
    left_on="Fecha_Diaria",
    right_on="Fecha",
    how="inner"
)

# Eliminar la fecha duplicada de IDEAM
df_consolidado = df_consolidado.drop(columns="Fecha")

display(df_consolidado.head())
display(df_consolidado.tail())

# Diagnóstico de días secos en el conjunto consolidado
dias_secos = (df_consolidado["Valor"] == 0).sum()
dias_totales = len(df_consolidado)
porcentaje_secos = dias_secos / dias_totales * 100

print(f"Días sin precipitación registrada (Valor = 0 mm): {dias_secos} de {dias_totales} ({porcentaje_secos:.2f}%)")

,Fecha_Diaria,tp,Valor
0,1952-06-01,46.130993,0.0
1,1952-06-02,3.012763,0.0
2,1952-06-03,5.734891,0.0
3,1952-06-04,18.395502,0.0
4,1952-06-05,24.197504,0.0


,Fecha_Diaria,tp,Valor
23260,2026-08-22,2.588633,0.3
23261,2026-08-23,8.760862,34.4
23262,2026-08-25,11.366934,2.0
23263,2026-08-30,0.652130,1.8
23264,2026-08-31,18.834227,0.5


Días sin precipitación registrada (Valor = 0 mm): 18595 de 23265 (79.93%)


In [6]:
# ============================================
# CELDA 6 — PREPARACIÓN DEL DATASET
# ============================================

# Orden cronológico
df_modelado = (
    df_consolidado
    .dropna(subset=["tp", "Valor"])
    .sort_values("Fecha_Diaria")
    .reset_index(drop=True)
)

display(df_modelado.head())
display(df_modelado.tail())

print(f"Registros disponibles: {len(df_modelado):,}")

,Fecha_Diaria,tp,Valor
0,1952-06-01,46.130993,0.0
1,1952-06-02,3.012763,0.0
2,1952-06-03,5.734891,0.0
3,1952-06-04,18.395502,0.0
4,1952-06-05,24.197504,0.0


,Fecha_Diaria,tp,Valor
23260,2026-08-22,2.588633,0.3
23261,2026-08-23,8.760862,34.4
23262,2026-08-25,11.366934,2.0
23263,2026-08-30,0.652130,1.8
23264,2026-08-31,18.834227,0.5


Registros disponibles: 23,265


In [7]:
# ============================================
# CELDA 7 — DIVISIÓN ENTRENAMIENTO / PRUEBA
# ============================================

from sklearn.model_selection import train_test_split

n = len(df_modelado)

df_train, df_test = train_test_split(
    df_modelado,
    train_size=0.85,
    shuffle=True,
    random_state=42
)

df_train = df_train.copy()
df_test = df_test.copy()

print(f"Entrenamiento: {len(df_train):,} ({len(df_train)/n:.0%})")
print(f"Prueba:        {len(df_test):,} ({len(df_test)/n:.0%})")

Entrenamiento: 19,775 (85%)
Prueba:        3,490 (15%)


In [8]:
# ============================================
# CELDA 8 — CORRECCIÓN DE SESGO MEDIANTE QUANTILE MAPPING
# ============================================

import numpy as np

# Función para aprender y aplicar Quantile Mapping
def aplicar_qm(tp_referencia, valor_referencia, tp_objetivo):

    valores_ideam = np.sort(valor_referencia)

    percentiles_objetivo = np.searchsorted(
        np.sort(tp_referencia),
        tp_objetivo,
        side="right"
    ) / (len(tp_referencia) + 1)

    percentiles_objetivo = np.clip(
        percentiles_objetivo,
        1 / (len(tp_referencia) + 1),
        len(tp_referencia) / (len(tp_referencia) + 1)
    )

    return np.quantile(valores_ideam, percentiles_objetivo)

# Quantile Mapping aprendido únicamente con entrenamiento
df_train = df_train.copy()
df_test = df_test.copy()

df_train["tp_qm"] = aplicar_qm(
    df_train["tp"].to_numpy(),
    df_train["Valor"].to_numpy(),
    df_train["tp"].to_numpy()
)

df_test["tp_qm"] = aplicar_qm(
    df_train["tp"].to_numpy(),
    df_train["Valor"].to_numpy(),
    df_test["tp"].to_numpy()
)

# Comprobación
print("ENTRENAMIENTO")
display(df_train[["Fecha_Diaria", "tp", "Valor", "tp_qm"]].head())

print("PRUEBA")
display(df_test[["Fecha_Diaria", "tp", "Valor", "tp_qm"]].head())

print(f"\nRegistros entrenamiento: {len(df_train):,}")
print(f"Registros prueba:       {len(df_test):,}")

ENTRENAMIENTO


,Fecha_Diaria,tp,Valor,tp_qm
16249,2007-03-07,0.625847,0.0,0.0
11551,1993-12-19,0.624913,0.0,0.0
21756,2022-07-02,3.266759,0.0,0.0
21640,2022-03-08,0.059079,0.0,0.0
21138,2020-09-24,8.650086,0.0,0.2


PRUEBA


,Fecha_Diaria,tp,Valor,tp_qm
4985,1974-11-27,22.120121,0.0,5.21068
9237,1987-08-18,6.005645,0.0,0.00000
2602,1960-10-17,14.950948,0.0,1.80000
6916,1980-05-11,0.987574,0.0,0.00000
11702,1994-05-19,7.595629,0.0,0.00000



Registros entrenamiento: 19,775
Registros prueba:       3,490


In [9]:
# ============================================
# CELDA 9 — VARIABLES PARA MODELADO
# ============================================

X_train = df_train[["tp_qm"]].copy()
y_train = df_train["Valor"].copy()

X_test = df_test[["tp_qm"]].copy()
y_test = df_test["Valor"].copy()

print(f"Observaciones entrenamiento: {len(X_train):,}")
print(f"Observaciones prueba:       {len(X_test):,}")
print("Predictor: tp_qm")
print("Variable objetivo: Valor")

Observaciones entrenamiento: 19,775
Observaciones prueba:       3,490
Predictor: tp_qm
Variable objetivo: Valor


In [10]:
# ============================================
# CELDA 10 — MÉTRICAS DE EVALUACIÓN
# ============================================

import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

# NSE
def nse(y_true, y_pred):
    denominador = np.sum((y_true - np.mean(y_true)) ** 2)

    if denominador == 0:
        return np.nan

    return 1 - np.sum((y_true - y_pred) ** 2) / denominador

# Evaluación de modelos
def evaluar_modelo(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "NSE": nse(y_true, y_pred)
    }

print("Métrica principal: RMSE")
print("Métricas complementarias: MAE y NSE")

Métrica principal: RMSE
Métricas complementarias: MAE y NSE


In [11]:
# ============================================
# CELDA 11 — VALIDACIÓN CRUZADA
# ============================================

from sklearn.model_selection import KFold

cv = KFold(n_splits=5, shuffle=True, random_state=42)

print("Validación cruzada: 5 folds")
print("Conjunto utilizado: entrenamiento (85 %)")

for i, (train_idx, val_idx) in enumerate(cv.split(X_train), start=1):
    print(
        f"Fold {i}: Entrenamiento = {len(train_idx):,} | "
        f"Validación = {len(val_idx):,}"
    )

# ============================================
# PRECÓMPUTO: QM por fold
# ============================================

folds_qm = []

for train_idx, val_idx in cv.split(X_train):

    tp_fold_train    = df_train["tp"].to_numpy()[train_idx]
    valor_fold_train = df_train["Valor"].to_numpy()[train_idx]

    tp_qm_fold_train = aplicar_qm(tp_fold_train, valor_fold_train, tp_fold_train)
    tp_qm_fold_val    = aplicar_qm(
        tp_fold_train, valor_fold_train,
        df_train["tp"].to_numpy()[val_idx]
    )

    folds_qm.append({
        "X_train": tp_qm_fold_train.reshape(-1, 1),
        "X_val":   tp_qm_fold_val.reshape(-1, 1),
        "y_train": y_train.to_numpy()[train_idx],
        "y_val":   y_train.to_numpy()[val_idx],
    })

Validación cruzada: 5 folds
Conjunto utilizado: entrenamiento (85 %)
Fold 1: Entrenamiento = 15,820 | Validación = 3,955
Fold 2: Entrenamiento = 15,820 | Validación = 3,955
Fold 3: Entrenamiento = 15,820 | Validación = 3,955
Fold 4: Entrenamiento = 15,820 | Validación = 3,955
Fold 5: Entrenamiento = 15,820 | Validación = 3,955


In [12]:
# ============================================
# CELDA 12 — MODELO BASE: ERA5-LAND + QUANTILE MAPPING
# ============================================

y_pred_baseline = X_test["tp_qm"]

metricas_baseline = evaluar_modelo(y_test, y_pred_baseline)

print("Modelo base: ERA5-Land + Quantile Mapping")
print(f"RMSE: {metricas_baseline['RMSE']:.3f} mm")
print(f"MAE : {metricas_baseline['MAE']:.3f} mm")
print(f"NSE : {metricas_baseline['NSE']:.4f}")

Modelo base: ERA5-Land + Quantile Mapping
RMSE: 8.438 mm
MAE : 2.170 mm
NSE : -0.5896


In [14]:
# ============================================
# CELDA 13 — MODELO 1: RANDOM FOREST + GA 
# ============================================

import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# --------------------------------------------
# 1. ESPACIO DE BÚSQUEDA
# --------------------------------------------

espacio = {
    "n_estimators": (100, 1000),
    "max_depth": (10, 100),
    "min_samples_split": (2, 10),
    "min_samples_leaf": (1, 10),
    "max_features": (0.1, 1.0),
    "max_samples": (0.1, 1.0)
}

pm_RF = 1 / len(espacio)   # 6 hiperparámetros → pm = 0.167

rng = np.random.default_rng(42)

# --------------------------------------------
# 2. CREAR INDIVIDUO
# --------------------------------------------

def crear_individuo():
    return [
        rng.integers(*espacio["n_estimators"], endpoint=True),
        rng.integers(*espacio["max_depth"], endpoint=True),
        rng.integers(*espacio["min_samples_split"], endpoint=True),
        rng.integers(*espacio["min_samples_leaf"], endpoint=True),
        rng.uniform(*espacio["max_features"]),
        rng.uniform(*espacio["max_samples"])
    ]

# --------------------------------------------
# 3. FUNCIÓN DE FITNESS
# --------------------------------------------

def fitness(ind):
    modelo = RandomForestRegressor(
        n_estimators=int(ind[0]),
        max_depth=int(ind[1]),
        min_samples_split=int(ind[2]),
        min_samples_leaf=int(ind[3]),
        max_features=ind[4],
        max_samples=ind[5],
        random_state=42,
        n_jobs=-1
    )

    rmse = []

    for fold in folds_qm:
        modelo.fit(fold["X_train"], fold["y_train"])
        pred = modelo.predict(fold["X_val"])
        rmse.append(np.sqrt(mean_squared_error(fold["y_val"], pred)))

    return np.mean(rmse)

# --------------------------------------------
# 4. CONFIGURACIÓN DEL GA
# --------------------------------------------

poblacion_n = 50
generaciones = 20

poblacion = [
    crear_individuo()
    for _ in range(poblacion_n)
]

mejor_rmse = np.inf
mejor_individuo = None

# --------------------------------------------
# 5. EVOLUCIÓN
# --------------------------------------------

for _ in range(generaciones):

    evaluados = sorted(
        [(fitness(ind), ind) for ind in poblacion],
        key=lambda x: x[0]
    )

    if evaluados[0][0] < mejor_rmse:
        mejor_rmse = evaluados[0][0]
        mejor_individuo = evaluados[0][1].copy()

    padres = [
        ind for _, ind in evaluados[:4]
    ]

    nueva_poblacion = padres.copy()

    while len(nueva_poblacion) < poblacion_n:

        p1 = padres[rng.integers(4)]
        p2 = padres[rng.integers(4)]

        hijo = [
            p1[i] if rng.random() < 0.5 else p2[i]
            for i in range(6)
        ]

        if rng.random() < pm_RF:
            hijo[0] = rng.integers(*espacio["n_estimators"], endpoint=True)

        if rng.random() < pm_RF:
            hijo[1] = rng.integers(*espacio["max_depth"], endpoint=True)

        if rng.random() < pm_RF:
            hijo[2] = rng.integers(*espacio["min_samples_split"], endpoint=True)

        if rng.random() < pm_RF:
            hijo[3] = rng.integers(*espacio["min_samples_leaf"], endpoint=True)

        if rng.random() < pm_RF:
            hijo[4] = rng.uniform(*espacio["max_features"])

        if rng.random() < pm_RF:
            hijo[5] = rng.uniform(*espacio["max_samples"])

        nueva_poblacion.append(hijo)

    poblacion = nueva_poblacion

# --------------------------------------------
# 6. MEJORES HIPERPARÁMETROS
# --------------------------------------------

mejores_parametros = {
    "n_estimators": int(mejor_individuo[0]),
    "max_depth": int(mejor_individuo[1]),
    "min_samples_split": int(mejor_individuo[2]),
    "min_samples_leaf": int(mejor_individuo[3]),
    "max_features": mejor_individuo[4],
    "max_samples": mejor_individuo[5]
}

# --------------------------------------------
# 7. MODELO DEFINITIVO
# --------------------------------------------

modelo_rf = RandomForestRegressor(
    **mejores_parametros,
    random_state=42,
    n_jobs=-1
)

modelo_rf.fit(X_train, y_train)

# --------------------------------------------
# 8. EVALUACIÓN INDEPENDIENTE
# --------------------------------------------

y_pred_rf = np.clip(
    modelo_rf.predict(X_test),
    0,
    None
)

metricas_rf = evaluar_modelo(y_test, y_pred_rf)

# --------------------------------------------
# 9. REGISTRAR RESULTADOS
# --------------------------------------------

if "resultados_modelos" not in globals():
    resultados_modelos = []

resultados_modelos = [
    r for r in resultados_modelos
    if r["Modelo"] != "Random Forest"
]

resultados_modelos.append({
    "Modelo": "Random Forest",
    "RMSE": metricas_rf["RMSE"],
    "MAE": metricas_rf["MAE"],
    "NSE": metricas_rf["NSE"]
})

# --------------------------------------------
# 10. RESULTADO FINAL
# --------------------------------------------

print("=" * 60)
print("RANDOM FOREST — RESULTADO FINAL")
print("=" * 60)
print(f"Mejores hiperparámetros: {mejores_parametros}")
print(f"RMSE validación GA     : {mejor_rmse:.3f} mm")
print(f"RMSE test              : {metricas_rf['RMSE']:.3f} mm")
print(f"MAE test               : {metricas_rf['MAE']:.3f} mm")
print(f"NSE test               : {metricas_rf['NSE']:.4f}")
print("=" * 60)

RANDOM FOREST — RESULTADO FINAL
Mejores hiperparámetros: {'n_estimators': 966, 'max_depth': 13, 'min_samples_split': 8, 'min_samples_leaf': 10, 'max_features': 0.9720925608195387, 'max_samples': 0.12052348392672775}
RMSE validación GA     : 6.461 mm
RMSE test              : 6.502 mm
MAE test               : 2.107 mm
NSE test               : 0.0563


In [15]:
# ============================================
# CELDA 14 — MODELO 2: XGBOOST + GA
# ============================================

import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

# --------------------------------------------
# 1. ESPACIO DE BÚSQUEDA
# --------------------------------------------

espacio = {
    "n_estimators": (10, 500),
    "max_depth": (2, 10),
    "learning_rate": (0.01, 0.30),
    "subsample": (0.3, 1.0),
    "colsample_bytree": (0, 1.0),
    "min_child_weight": (0, 1.5),
    "gamma": (0, 1.0)
}

pm_XGB = 1 / len(espacio)   # 7 hiperparámetros → pm = 0.143

rng = np.random.default_rng(42)

# --------------------------------------------
# 2. CREAR INDIVIDUO
# --------------------------------------------

def crear_individuo():
    return [
        rng.integers(*espacio["n_estimators"], endpoint=True),
        rng.integers(*espacio["max_depth"], endpoint=True),
        rng.uniform(*espacio["learning_rate"]),
        rng.uniform(*espacio["subsample"]),
        rng.uniform(*espacio["colsample_bytree"]),
        rng.uniform(*espacio["min_child_weight"]),
        rng.uniform(*espacio["gamma"])
    ]

# --------------------------------------------
# 3. FUNCIÓN DE FITNESS
# --------------------------------------------

def fitness(ind):
    modelo = XGBRegressor(
        n_estimators=int(ind[0]),
        max_depth=int(ind[1]),
        learning_rate=ind[2],
        subsample=ind[3],
        colsample_bytree=ind[4],
        min_child_weight=ind[5],
        gamma=ind[6],
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    rmse = []

    for fold in folds_qm:
        modelo.fit(fold["X_train"], fold["y_train"])
        pred = modelo.predict(fold["X_val"])
        rmse.append(np.sqrt(mean_squared_error(fold["y_val"], pred)))

    return np.mean(rmse)

# --------------------------------------------
# 4. CONFIGURACIÓN DEL GA
# --------------------------------------------

poblacion_n = 50
generaciones = 20

poblacion = [
    crear_individuo()
    for _ in range(poblacion_n)
]

mejor_rmse = np.inf
mejor_individuo = None

# --------------------------------------------
# 5. EVOLUCIÓN
# --------------------------------------------

for _ in range(generaciones):

    evaluados = sorted(
        [(fitness(ind), ind) for ind in poblacion],
        key=lambda x: x[0]
    )

    if evaluados[0][0] < mejor_rmse:
        mejor_rmse = evaluados[0][0]
        mejor_individuo = evaluados[0][1].copy()

    padres = [
        ind for _, ind in evaluados[:4]
    ]

    nueva_poblacion = padres.copy()

    while len(nueva_poblacion) < poblacion_n:

        p1 = padres[rng.integers(4)]
        p2 = padres[rng.integers(4)]

        hijo = [
            p1[i] if rng.random() < 0.5 else p2[i]
            for i in range(7)
        ]

        if rng.random() < pm_XGB:
            hijo[0] = rng.integers(*espacio["n_estimators"], endpoint=True)

        if rng.random() < pm_XGB:
            hijo[1] = rng.integers(*espacio["max_depth"], endpoint=True)

        if rng.random() < pm_XGB:
            hijo[2] = rng.uniform(*espacio["learning_rate"])

        if rng.random() < pm_XGB:
            hijo[3] = rng.uniform(*espacio["subsample"])

        if rng.random() < pm_XGB:
            hijo[4] = rng.uniform(*espacio["colsample_bytree"])

        if rng.random() < pm_XGB:
            hijo[5] = rng.uniform(*espacio["min_child_weight"])

        if rng.random() < pm_XGB:
            hijo[6] = rng.uniform(*espacio["gamma"])

        nueva_poblacion.append(hijo)

    poblacion = nueva_poblacion

# --------------------------------------------
# 6. MEJORES HIPERPARÁMETROS
# --------------------------------------------

mejores_parametros = {
    "n_estimators": int(mejor_individuo[0]),
    "max_depth": int(mejor_individuo[1]),
    "learning_rate": mejor_individuo[2],
    "subsample": mejor_individuo[3],
    "colsample_bytree": mejor_individuo[4],
    "min_child_weight": mejor_individuo[5],
    "gamma": mejor_individuo[6]
}

# --------------------------------------------
# 7. MODELO DEFINITIVO
# --------------------------------------------

modelo_xgb = XGBRegressor(
    **mejores_parametros,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

modelo_xgb.fit(X_train, y_train)

# --------------------------------------------
# 8. EVALUACIÓN INDEPENDIENTE
# --------------------------------------------

y_pred_xgb = np.clip(
    modelo_xgb.predict(X_test),
    0,
    None
)

metricas_xgb = evaluar_modelo(
    y_test,
    y_pred_xgb
)

# --------------------------------------------
# 9. REGISTRAR RESULTADOS
# --------------------------------------------

if "resultados_modelos" not in globals():
    resultados_modelos = []

resultados_modelos = [
    r for r in resultados_modelos
    if r["Modelo"] != "XGBoost"
]

resultados_modelos.append({
    "Modelo": "XGBoost",
    "RMSE": metricas_xgb["RMSE"],
    "MAE": metricas_xgb["MAE"],
    "NSE": metricas_xgb["NSE"]
})

# --------------------------------------------
# 10. RESULTADO FINAL
# --------------------------------------------

print("=" * 60)
print("XGBOOST — RESULTADO FINAL")
print("=" * 60)
print(f"Mejores hiperparámetros: {mejores_parametros}")
print(f"RMSE validación GA : {mejor_rmse:.3f} mm")
print(f"RMSE test          : {metricas_xgb['RMSE']:.3f} mm")
print(f"MAE test           : {metricas_xgb['MAE']:.3f} mm")
print(f"NSE test           : {metricas_xgb['NSE']:.4f}")
print("=" * 60)


XGBOOST — RESULTADO FINAL
Mejores hiperparámetros: {'n_estimators': 52, 'max_depth': 2, 'learning_rate': 0.039841862524854416, 'subsample': 0.5692784308629389, 'colsample_bytree': 0.5081673558128751, 'min_child_weight': 1.277947553402386, 'gamma': 0.579219568941896}
RMSE validación GA : 6.441 mm
RMSE test          : 6.486 mm
MAE test           : 2.123 mm
NSE test           : 0.0608


In [16]:
# ============================================
# CELDA 15 — MODELO 3: EXTRA TREES + GA
# ============================================

import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_squared_error

# --------------------------------------------
# 1. ESPACIO DE BÚSQUEDA
# --------------------------------------------

espacio = {
    "n_estimators": (100, 1000),
    "max_depth": (10, 100),
    "min_samples_split": (2, 10),
    "min_samples_leaf": (1, 10),
    "max_features": (0.1, 1.0)
}

pm_ET = 1 / len(espacio)   # 5 hiperparámetros → pm = 0.200

rng = np.random.default_rng(42)

# --------------------------------------------
# 2. CREAR INDIVIDUO
# --------------------------------------------

def crear_individuo():
    return [
        rng.integers(*espacio["n_estimators"], endpoint=True),
        rng.integers(*espacio["max_depth"], endpoint=True),
        rng.integers(*espacio["min_samples_split"], endpoint=True),
        rng.integers(*espacio["min_samples_leaf"], endpoint=True),
        rng.uniform(*espacio["max_features"])
    ]

# --------------------------------------------
# 3. FUNCIÓN DE FITNESS
# --------------------------------------------

def fitness(ind):
    modelo = ExtraTreesRegressor(
        n_estimators=int(ind[0]),
        max_depth=int(ind[1]),
        min_samples_split=int(ind[2]),
        min_samples_leaf=int(ind[3]),
        max_features=ind[4],
        random_state=42,
        n_jobs=-1
    )

    rmse = []

    for fold in folds_qm:
        modelo.fit(fold["X_train"], fold["y_train"])
        pred = modelo.predict(fold["X_val"])
        rmse.append(np.sqrt(mean_squared_error(fold["y_val"], pred)))

    return np.mean(rmse)

# --------------------------------------------
# 4. CONFIGURACIÓN DEL GA
# --------------------------------------------

poblacion_n = 50
generaciones = 20

poblacion = [
    crear_individuo()
    for _ in range(poblacion_n)
]

mejor_rmse = np.inf
mejor_individuo = None

# --------------------------------------------
# 5. EVOLUCIÓN
# --------------------------------------------

for _ in range(generaciones):

    evaluados = sorted(
        [(fitness(ind), ind) for ind in poblacion],
        key=lambda x: x[0]
    )

    if evaluados[0][0] < mejor_rmse:
        mejor_rmse = evaluados[0][0]
        mejor_individuo = evaluados[0][1].copy()

    padres = [
        ind for _, ind in evaluados[:4]
    ]

    nueva_poblacion = padres.copy()

    while len(nueva_poblacion) < poblacion_n:

        p1 = padres[rng.integers(4)]
        p2 = padres[rng.integers(4)]

        hijo = [
            p1[i] if rng.random() < 0.5 else p2[i]
            for i in range(5)
        ]

        if rng.random() < pm_ET:
            hijo[0] = rng.integers(*espacio["n_estimators"], endpoint=True)

        if rng.random() < pm_ET:
            hijo[1] = rng.integers(*espacio["max_depth"], endpoint=True)

        if rng.random() < pm_ET:
            hijo[2] = rng.integers(*espacio["min_samples_split"], endpoint=True)

        if rng.random() < pm_ET:
            hijo[3] = rng.integers(*espacio["min_samples_leaf"], endpoint=True)

        if rng.random() < pm_ET:
            hijo[4] = rng.uniform(*espacio["max_features"])

        nueva_poblacion.append(hijo)

    poblacion = nueva_poblacion

# --------------------------------------------
# 6. MEJORES HIPERPARÁMETROS
# --------------------------------------------

mejores_parametros = {
    "n_estimators": int(mejor_individuo[0]),
    "max_depth": int(mejor_individuo[1]),
    "min_samples_split": int(mejor_individuo[2]),
    "min_samples_leaf": int(mejor_individuo[3]),
    "max_features": mejor_individuo[4]
}

# --------------------------------------------
# 7. MODELO DEFINITIVO
# --------------------------------------------

modelo_et = ExtraTreesRegressor(
    **mejores_parametros,
    random_state=42,
    n_jobs=-1
)

modelo_et.fit(X_train, y_train)

# --------------------------------------------
# 8. EVALUACIÓN INDEPENDIENTE
# --------------------------------------------

y_pred_et = np.clip(
    modelo_et.predict(X_test),
    0,
    None
)

metricas_et = evaluar_modelo(
    y_test,
    y_pred_et
)

# --------------------------------------------
# 9. REGISTRAR RESULTADOS
# --------------------------------------------

if "resultados_modelos" not in globals():
    resultados_modelos = []

resultados_modelos = [
    r for r in resultados_modelos
    if r["Modelo"] != "Extra Trees"
]

resultados_modelos.append({
    "Modelo": "Extra Trees",
    "RMSE": metricas_et["RMSE"],
    "MAE": metricas_et["MAE"],
    "NSE": metricas_et["NSE"]
})

# --------------------------------------------
# 10. RESULTADO FINAL
# --------------------------------------------

print("=" * 60)
print("EXTRA TREES — RESULTADO FINAL")
print("=" * 60)
print(f"Mejores hiperparámetros: {mejores_parametros}")
print(f"RMSE validación GA : {mejor_rmse:.3f} mm")
print(f"RMSE test          : {metricas_et['RMSE']:.3f} mm")
print(f"MAE test           : {metricas_et['MAE']:.3f} mm")
print(f"NSE test           : {metricas_et['NSE']:.4f}")
print("=" * 60)


EXTRA TREES — RESULTADO FINAL
Mejores hiperparámetros: {'n_estimators': 420, 'max_depth': 10, 'min_samples_split': 6, 'min_samples_leaf': 4, 'max_features': 0.5340730932486102}
RMSE validación GA : 6.453 mm
RMSE test          : 6.512 mm
MAE test           : 2.139 mm
NSE test           : 0.0534


In [17]:
# ============================================
# CELDA 16 — MODELO 4: LIGHTGBM + GA
# ============================================

import numpy as np
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

# --------------------------------------------
# 1. ESPACIO DE BÚSQUEDA
# --------------------------------------------

espacio = {
    "max_depth": (6, 10),
    "learning_rate": (0.02, 0.10),
    "num_leaves": (20, 500),
    "min_child_samples": (20, 1000)
}

N_ESTIMATORS_LGBM = 400   # Valor fijo, que no forma parte del genoma

pm_LGBM = 1 / len(espacio)   # 4 hiperparámetros tuneados → pm = 0.250

# --------------------------------------------
# 2. CREAR INDIVIDUO
# --------------------------------------------

def crear_individuo():
    return [
        rng.integers(*espacio["max_depth"], endpoint=True),
        rng.uniform(*espacio["learning_rate"]),
        rng.integers(*espacio["num_leaves"], endpoint=True),
        rng.integers(*espacio["min_child_samples"], endpoint=True)
    ]

# --------------------------------------------
# 3. FUNCIÓN DE FITNESS
# --------------------------------------------

def fitness(ind):
    modelo = LGBMRegressor(
        n_estimators=N_ESTIMATORS_LGBM,
        max_depth=int(ind[0]),
        learning_rate=ind[1],
        num_leaves=int(ind[2]),
        min_child_samples=int(ind[3]),
        objective="regression",
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    rmse = []

    for fold in folds_qm:
        modelo.fit(fold["X_train"], fold["y_train"])
        pred = modelo.predict(fold["X_val"])
        rmse.append(np.sqrt(mean_squared_error(fold["y_val"], pred)))

    return np.mean(rmse)
    
# --------------------------------------------
# 4. CONFIGURACIÓN DEL GA
# --------------------------------------------

poblacion_n = 50
generaciones = 20

poblacion = [
    crear_individuo()
    for _ in range(poblacion_n)
]

mejor_rmse = np.inf
mejor_individuo = None

# --------------------------------------------
# 5. EVOLUCIÓN
# --------------------------------------------

for _ in range(generaciones):

    evaluados = sorted(
        [(fitness(ind), ind) for ind in poblacion],
        key=lambda x: x[0]
    )

    if evaluados[0][0] < mejor_rmse:
        mejor_rmse = evaluados[0][0]
        mejor_individuo = evaluados[0][1].copy()

    padres = [
        ind for _, ind in evaluados[:4]
    ]

    nueva_poblacion = padres.copy()

    while len(nueva_poblacion) < poblacion_n:

        p1 = padres[rng.integers(4)]
        p2 = padres[rng.integers(4)]

        hijo = [
            p1[i] if rng.random() < 0.5 else p2[i]
            for i in range(4)
        ]

        if rng.random() < pm_LGBM:
            hijo[0] = rng.integers(*espacio["max_depth"], endpoint=True)

        if rng.random() < pm_LGBM:
            hijo[1] = rng.uniform(*espacio["learning_rate"])

        if rng.random() < pm_LGBM:
            hijo[2] = rng.integers(*espacio["num_leaves"], endpoint=True)

        if rng.random() < pm_LGBM:
            hijo[3] = rng.integers(*espacio["min_child_samples"], endpoint=True)

        nueva_poblacion.append(hijo)

    poblacion = nueva_poblacion

# --------------------------------------------
# 6. MEJORES HIPERPARÁMETROS
# --------------------------------------------

mejores_parametros = {
    "n_estimators": N_ESTIMATORS_LGBM,
    "max_depth": int(mejor_individuo[0]),
    "learning_rate": mejor_individuo[1],
    "num_leaves": int(mejor_individuo[2]),
    "min_child_samples": int(mejor_individuo[3])
}

# --------------------------------------------
# 7. MODELO DEFINITIVO
# --------------------------------------------

modelo_lgbm = LGBMRegressor(
    **mejores_parametros,
    objective="regression",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

modelo_lgbm.fit(
    X_train,
    y_train
)

# --------------------------------------------
# 8. EVALUACIÓN INDEPENDIENTE
# --------------------------------------------

y_pred_lgbm = np.clip(
    modelo_lgbm.predict(X_test),
    0,
    None
)

metricas_lgbm = evaluar_modelo(
    y_test,
    y_pred_lgbm
)

# --------------------------------------------
# 9. REGISTRAR RESULTADOS
# --------------------------------------------

if "resultados_modelos" not in globals():
    resultados_modelos = []

resultados_modelos = [
    r for r in resultados_modelos
    if r["Modelo"] != "LightGBM"
]

resultados_modelos.append({
    "Modelo": "LightGBM",
    "RMSE": metricas_lgbm["RMSE"],
    "MAE": metricas_lgbm["MAE"],
    "NSE": metricas_lgbm["NSE"]
})

# --------------------------------------------
# 10. RESULTADO FINAL
# --------------------------------------------

print("=" * 60)
print("LIGHTGBM — RESULTADO FINAL")
print("=" * 60)
print(f"Mejores hiperparámetros: {mejores_parametros}")
print(f"RMSE validación GA : {mejor_rmse:.3f} mm")
print(f"RMSE test          : {metricas_lgbm['RMSE']:.3f} mm")
print(f"MAE test           : {metricas_lgbm['MAE']:.3f} mm")
print(f"NSE test           : {metricas_lgbm['NSE']:.4f}")
print("=" * 60)

LIGHTGBM — RESULTADO FINAL
Mejores hiperparámetros: {'n_estimators': 400, 'max_depth': 9, 'learning_rate': 0.021316136256361293, 'num_leaves': 293, 'min_child_samples': 944}
RMSE validación GA : 6.438 mm
RMSE test          : 6.488 mm
MAE test           : 2.106 mm
NSE test           : 0.0605


In [ ]:
# ============================================
# CELDA 17 — ENSAMBLE MEDIANTE PROMEDIO SIMPLE CON PONDERACIÓN UNIFORME
# ============================================

import numpy as np

# Predicciones de los cuatro modelos
predicciones = np.column_stack([
    y_pred_rf,
    y_pred_xgb,
    y_pred_et,
    y_pred_lgbm
])

# Promedio simple de las predicciones
y_pred_ensemble = np.mean(
    predicciones,
    axis=1
)

# Asegurar precipitación no negativa
y_pred_ensemble = np.clip(
    y_pred_ensemble,
    0,
    None
)

# Evaluación del ensamble
metricas_ensemble = evaluar_modelo(
    y_test,
    y_pred_ensemble
)

# Registrar resultados
if "resultados_modelos" not in globals():
    resultados_modelos = []

resultados_modelos = [
    r for r in resultados_modelos
    if r["Modelo"] != "Ensemble"
]

resultados_modelos.append({
    "Modelo": "Ensemble",
    "RMSE": metricas_ensemble["RMSE"],
    "MAE": metricas_ensemble["MAE"],
    "NSE": metricas_ensemble["NSE"]
})

# Resultado final
print("=" * 60)
print("ENSEMBLE — PROMEDIO SIMPLE")
print("=" * 60)
print("Modelos combinados : Random Forest, XGBoost, Extra Trees, LightGBM")
print("Pesos              : Iguales (0.25 cada uno)")
print(f"RMSE test          : {metricas_ensemble['RMSE']:.3f} mm")
print(f"MAE test           : {metricas_ensemble['MAE']:.3f} mm")
print(f"NSE test           : {metricas_ensemble['NSE']:.4f}")
print("=" * 60)

ENSEMBLE — PROMEDIO SIMPLE
Modelos combinados : Random Forest, XGBoost, Extra Trees, LightGBM
Pesos              : Iguales (0.25 cada uno)
RMSE test          : 6.491 mm
MAE test           : 2.117 mm
NSE test           : 0.0595
